# 12 · Vectorization and measured performance

**Level:** Advanced · **Practice time:** 30–50 minutes

[← Previous](11_copies_views_and_memory.ipynb) · [Course guide](../README.md) · [Next →](13_numerical_stability.ipynb)

**You will learn:** Replace loops thoughtfully, measure runtime fairly, and limit temporary memory.

**Prerequisites:** Notebooks 00–11.

## How to learn with this notebook

1. Run cells from top to bottom. Predict results before executing each example.
2. Do the **Try it** experiment by editing the example or adding a cell below it.
3. Replace `None` in each exercise with your answer, then rerun its check cell.
4. Consult the solutions at the end only after trying. An untouched exercise prints **Pending**, so Run All succeeds without claiming you solved it.
5. Complete the independent challenge, then restart the kernel and run all cells again.

Each notebook imports its own dependencies and creates its own data. You do not need variables from another notebook. The supplied outputs show a verified reference run; rerun cells to explore your changes.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
print("NumPy version:", np.__version__)

NumPy version: 2.5.3


## Step 1 · Establish correctness before timing

Vectorization moves repeated numeric work into array operations. Keep a small, clear reference implementation to verify the faster version. `np.vectorize` is a convenience wrapper around repeated Python calls, not a general performance optimization.

In [2]:
def squared_distance_loop(a, b):
    total = 0.0
    for i in range(len(a)):
        total += (a[i] - b[i]) ** 2
    return total

def squared_distance_numpy(a, b):
    return np.sum((a - b) ** 2)

rng = np.random.default_rng(4)
a, b = rng.normal(size=(2, 20_000))
np.testing.assert_allclose(squared_distance_loop(a, b), squared_distance_numpy(a, b))
print("Agreement checked.")

Agreement checked.


**Try it:** Check both functions on equal vectors and on one-element vectors.

## Step 2 · Benchmark repeated runs

Use repeated timings on the same inputs and report the setup. A minimum timing approximates a less interrupted run; real workloads may differ. Small arrays may not benefit from vectorization. Hardware, dtype, memory layout, and library builds influence results, so speed is not an assertion.

In [3]:
from timeit import repeat
loop_seconds = min(repeat(lambda: squared_distance_loop(a, b), number=3, repeat=3)) / 3
numpy_seconds = min(repeat(lambda: squared_distance_numpy(a, b), number=3, repeat=3)) / 3
print(f"Python loop: {loop_seconds:.6f} seconds/call")
print(f"NumPy:       {numpy_seconds:.6f} seconds/call")
print(f"Observed ratio: {loop_seconds / numpy_seconds:.1f}x")
print("Input shape/dtype:", a.shape, a.dtype)

Python loop: 0.006254 seconds/call
NumPy:       0.000018 seconds/call
Observed ratio: 350.0x
Input shape/dtype: (20000,) float64


**Try it:** Repeat for 10 values and 100,000 values; do not assume the same speed ratio.

## Step 3 · Reuse buffers and process chunks

Array expressions can create large temporaries. Ufunc `out=` lets you reuse storage. Chunking limits intermediate size. Here squared differences are computed a chunk at a time; the sum of chunk results matches the reference within floating point tolerance.

In [4]:
scratch = np.empty_like(a)
np.subtract(a, b, out=scratch)
np.square(scratch, out=scratch)
buffer_result = scratch.sum()
chunk_result = 0.0
chunk_size = 2048
for start in range(0, a.size, chunk_size):
    difference = a[start:start + chunk_size] - b[start:start + chunk_size]
    chunk_result += np.sum(difference ** 2)
np.testing.assert_allclose(buffer_result, chunk_result)
print("Full scratch bytes:", scratch.nbytes)
print("One chunk difference bytes (maximum):", chunk_size * a.itemsize)

Full scratch bytes: 160000
One chunk difference bytes (maximum): 16384


**Try it:** Explain why repeatedly calling np.append in a loop can repeatedly allocate and copy data.

## Your practice exercises

Use the named answer variables. The checks verify your result; they do not require the same approach as the reference solution. If an assertion fails, read the expected value and inspect your shape and dtype.

### Exercise 1

Compute the sum of squares for each row without a Python loop.

In [5]:
features = np.array([[1., 2., 3.], [4., 5., 6.]])

row_energy = None  # TODO: replace None with your answer

In [6]:
if row_energy is None:
    print("Pending: complete Exercise 1 first.")
else:
    np.testing.assert_allclose(row_energy, [14, 77])
    print("Exercise 1 passed!")

Pending: complete Exercise 1 first.


### Exercise 2

Implement f(x)=3*x+2 using array operations.

In [7]:
inputs = np.arange(5, dtype=float)

transformed = None  # TODO: replace None with your answer

In [8]:
if transformed is None:
    print("Pending: complete Exercise 2 first.")
else:
    np.testing.assert_allclose(transformed, [2, 5, 8, 11, 14])
    print("Exercise 2 passed!")

Pending: complete Exercise 2 first.


## Independent challenge

Benchmark row-wise Euclidean norms using a loop and a vectorized expression. Check correctness, state shape/dtype, and explain temporary memory costs.

Add code and explanation cells below. Decide on expected results before checking your implementation.

In [9]:
# Your independent challenge work goes here.

## Before you move on

- Explain the main idea in your own words without reading the examples.
- Make both exercise checks print **passed**.
- Try at least one changed input or edge case and explain the result.
- Finish the independent challenge or write down a specific question to revisit.

Expertise comes from repeated practice and debugging unfamiliar problems. Completing these notebooks gives you a broad practical foundation, not a substitute for that experience.

---

## Reference solutions — try first

These cells use separate reference variables, so they do not fill in your exercise answers. They also check the reference result. Compare your reasoning as well as the output.

### Solution 1

Compute the sum of squares for each row without a Python loop.

**Why it works:** Square each feature value and sum across columns, leaving one energy value per row.

In [10]:
reference_row_energy = (features ** 2).sum(axis=1)
np.testing.assert_allclose(reference_row_energy, [14, 77])
print(reference_row_energy)

[14. 77.]


### Solution 2

Implement f(x)=3*x+2 using array operations.

**Why it works:** Scalar multiplication and addition broadcast across the entire input array.

In [11]:
reference_transformed = 3 * inputs + 2
np.testing.assert_allclose(reference_transformed, [2, 5, 8, 11, 14])
print(reference_transformed)

[ 2.  5.  8. 11. 14.]


## Continue learning

- [Official NumPy documentation: ufuncs ](https://numpy.org/doc/stable/reference/ufuncs.html)
- [Official NumPy documentation: numpy.vectorize ](https://numpy.org/doc/stable/reference/generated/numpy.vectorize.html)

[Next →](13_numerical_stability.ipynb)